# Notebook 3 — Extração de Features (v5)

## Posição no pipeline

```
NB1  → data/signals/*_signal.npz          (sinal filtrado e reamostrado)
NB2  → data/level1_signals/*_L1_w{N}.npz (rótulos: interictal/pré-ictal/ictal/pós-ictal)
NB3  → data/features/{nível}/*__w{N}.npz  (vetores de features — ESTE NOTEBOOK)
NB4  → modelos ML personalizados por paciente
```

## O que este notebook faz

Para cada gravação EDF processada pelo NB1/NB2, extrai um vetor de **19 features por canal**
em janelas de 30 segundos (passo de 15s). O resultado é salvo por gravação individual —
não por paciente agregado — para que o NB4 possa fazer o split treino/teste por crise
sem precisar reprocessar nada.

## Janelas pré-ictais

O NB2 rotula cada gravação em **4 versões** (uma por janela candidata):
- `w10` — 10 minutos antes da crise = pré-ictal
- `w15` — 15 minutos antes da crise = pré-ictal
- `w30` — 30 minutos antes da crise = pré-ictal
- `w45` — 45 minutos antes da crise = pré-ictal

Este notebook extrai features para as 4 versões de rótulo, gerando 4 arquivos
`.npz` por gravação por nível. O NB4 escolhe qual janela usar em cada paciente
(janela ótima individual — Etapa 1 do NB4).

## Cascata de canais R5 → R0

As features são extraídas em **5 configurações de canais** (níveis), simulando
dispositivos com graus crescentes de simplicidade — de um EEG hospitalar completo
até um wearable de 2 eletrodos:

| Nível | N canais | Composição                                          | Features totais | Analogia clínica         |
|-------|----------|-----------------------------------------------------|-----------------|--------------------------|
| R5    | 17       | Frontal(7) + Temporal(4) + Central(2) + Parietal(2) + Occipital(2) | 323 | EEG hospitalar completo |
| R3    | 13       | Frontal(7) + Temporal(4) + Central(2)               | 247             | Sem eletrodos posteriores |
| R2    | 11       | Frontal(7) + Temporal(4)                            | 209             | EEG anterior e temporal  |
| R1    | 4        | Temporal(4)                                         | 76              | Capacete temporal mínimo |
| R0    | 2        | Par temporal behind-the-ear                         | 38              | Dispositivo vestível real |

**Propriedade de subconjunto:** R0 ⊆ R1 ⊆ R2 ⊆ R3 ⊆ R5. Cada nível é um subconjunto
exato do superior — reduzir canais significa sempre **remover**, nunca substituir.
Verificado por `assert` automático na importação do módulo.

## Otimização v5: calcular R5 uma vez, derivar o resto por fatiamento

Como R0⊆R1⊆R2⊆R3⊆R5, e os canais de cada nível ocupam colunas **contíguas** no
vetor R5 (verificado numericamente), não é necessário recalcular features para cada
nível separadamente. O código:

1. Calcula features para **R5** (17 canais × 19 = 323 features por janela)
2. Fatia o vetor R5 para obter R3, R2, R1, R0:

```
R5 → cols [0:323]    (todos os 17 canais)
R3 → cols [0:247]    (primeiros 13 canais: frontal+temporal+central)
R2 → cols [0:209]    (primeiros 11 canais: frontal+temporal)
R1 → cols [133:209]  (4 canais temporais — começam no canal 7 do R5: 7×19=133)
R0 → cols [152:190]  (CHBMIT: T7-FT9 e FT10-T8, canais 8-9 do R5)
    cols [133:171]   (Siena/Mendeley: T3 e T4, canais 7-8 do R5)
```

**Resultado:** redução de ~5088 tarefas pesadas para ~1016 — uma por (gravação × janela),
não por (gravação × janela × nível).

**SeizeIT2 é tratado separadamente** — seus canais (`t3`, `t4` behind-the-ear) são
completamente diferentes dos outros datasets e não participam da hierarquia R5→R0.
Só tem nível R0, processado diretamente.

## Retomada automática

A execução pode ser **interrompida e retomada a qualquer momento** sem perda de progresso.
A lógica de retomada verifica, para cada (gravação × janela), se **todos os 5 níveis**
já existem em disco. Se qualquer nível estiver faltando, a tarefa é reprocessada.
Arquivos já completos são pulados sem releitura.

## Estrutura de saída

```
data/features/
  R5/  CHBMIT__chb01__chb01_03__w30.npz   ← gravação chb01_03, janela 30min, nível R5
  R3/  CHBMIT__chb01__chb01_03__w30.npz   ← mesma gravação, nível R3 (fatiado de R5)
  R2/  ...
  R1/  ...
  R0/  ...
       SeizeIT2__sub-001__...__w30.npz     ← SeizeIT2 só em R0, processado diretamente
data/logs/
  nb3_manifest.csv                         ← índice de todos os arquivos gerados
```

Cada `.npz` contém: `X_pre`, `X_inter`, `t_pre`, `t_inter`, `channels`, `feat_names`,
`dataset`, `patient`, `fkey`, `level`, `window_min`, `file_order`, `n_seizures`,
`n_pre`, `n_inter`, `n_total`, `win_sec`, `step_sec`, `n_feat_per_ch`,
`col_start`, `col_end` (offsets de coluna no vetor R5, para auditoria).


## 1. Dependências

Instala as bibliotecas necessárias. Pode pular se já estiverem instaladas.


In [1]:
import os
%pip install -q numpy pandas scipy PyWavelets joblib tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Módulo `nb3_worker.py`

### Por que um arquivo `.py` separado?

No Windows, o `ProcessPoolExecutor` usa o método `spawn` para criar processos filhos:
cada filho inicia um interpretador Python do zero e **reimporta** o código que vai executar.
Código definido dentro de um notebook Jupyter vive no `__main__` do kernel — os processos
filhos não conseguem reimportá-lo e morrem com `BrokenProcessPool`.

A solução é escrever todo o código que roda nos filhos em um arquivo `.py` real (`nb3_worker.py`),
usando `%%writefile`. Como é um módulo de verdade em disco, o `spawn` consegue importá-lo
normalmente em cada processo filho.

### O que está no worker

- **Configuração:** pacientes, janelas `[10,15,30,45]`, níveis, composição de canais por dataset
- **`channel_feats(sig)`:** calcula as 19 features de um canal:
  - *Temporais (7):* desvio padrão, variância, RMS, comprimento de linha, mobilidade de Hjorth, assimetria, curtose
  - *Espectrais (7):* potência em delta/theta/alpha/beta/gamma, entropia espectral, potência relativa de beta
  - *Wavelet (4):* energia dos coeficientes de detalhe DWT db4 nos níveis d1–d4
  - *Complexidade (1):* complexidade de Hjorth
- **`build_fvec_r5(win_data, chs, ds)`:** aplica `channel_feats` aos 17 canais do R5 e concatena → vetor de 323 features
- **`build_fvec_seizeit2(win_data, chs)`:** aplica `channel_feats` aos 2 canais do SeizeIT2
- **`R5_SLICES`:** dict pré-calculado com os slices de coluna `{dataset: {nível: slice(a,b)}}`
- **`process_file_r5(ds, pat, fkey, w, fo)`:** função principal — carrega sinal+rótulos, percorre janelas, calcula R5, fatia para R3/R2/R1/R0, salva 5 arquivos
- **`process_file_seizeit2(pat, fkey, w, fo)`:** versão para SeizeIT2 — calcula R0 diretamente
- **`build_task_list_v5()`:** monta a lista de tarefas pendentes (com retomada automática)
- **`_save_level(...)`:** helper que salva um arquivo `.npz` com todos os metadados


In [2]:
%%writefile nb3_worker.py
'''Módulo de trabalho do NB3 v5 — otimizado para processar R5 uma vez e
derivar R3/R2/R1/R0 por fatiamento de colunas, em vez de recalcular features
para cada nível separadamente. Reduz o número de tarefas pesadas de 5x para 1x
(para CHBMIT/Siena/Mendeley). SeizeIT2 mantém fluxo próprio (só R0, canais
diferentes). Importado pelo notebook e pelos processos filhos do
ProcessPoolExecutor — precisa ser um arquivo .py real para funcionar no Windows.
'''
import os, re, glob, gc
import numpy as np
from scipy.signal import welch
from scipy.stats import skew, kurtosis
import pywt

np.trapz = getattr(np, 'trapz', getattr(np, 'trapezoid', None))

# ── Diretórios ──
ROOT_DIR   = 'data'
SIGNAL_DIR = os.path.join(ROOT_DIR, 'signals')
L1_DIR     = os.path.join(ROOT_DIR, 'level1_signals')
FEAT_DIR   = os.path.join(ROOT_DIR, 'features')
LOG_DIR    = os.path.join(ROOT_DIR, 'logs')
os.makedirs(LOG_DIR, exist_ok=True)

# ── Pacientes selecionados ──
PATIENTS = {
    'CHBMIT'  : ['chb01','chb03','chb04','chb05','chb06','chb07',
                 'chb08','chb10','chb11','chb12','chb13','chb14'],
    'Siena'   : ['PN00','PN01','PN03','PN05','PN06','PN09',
                 'PN10','PN12','PN13','PN14','PN16','PN17'],
    'Mendeley': ['p10','p11','p12','p13','p14','p15'],
    'SeizeIT2': ['sub-001','sub-002','sub-004','sub-005',
                 'sub-007','sub-008','sub-011','sub-012',
                 'sub-035','sub-039','sub-047','sub-073'],
}

# ── Janelas pré-ictais ──
PREICTAL_WINDOWS_MIN = [10, 15, 30, 45]

# ── Níveis de canal ──
LEVELS = ['R5', 'R3', 'R2', 'R1', 'R0']
LEVEL_DS = {
    'R5': ['CHBMIT','Siena','Mendeley'],
    'R3': ['CHBMIT','Siena','Mendeley'],
    'R2': ['CHBMIT','Siena','Mendeley'],
    'R1': ['CHBMIT','Siena','Mendeley'],
    'R0': ['CHBMIT','Siena','Mendeley','SeizeIT2'],
}

# ── Composição de canais por nível e dataset ──
FRONTAL_CH  = {'CHBMIT':['FP1-F7','F7-T7','FP1-F3','FP2-F4','FP2-F8','F8-T8','FZ-CZ'],
               'Siena':['Fp1','F3','F7','Fz','Fp2','F4','F8'],
               'Mendeley':['Fp1','Fp2','F3','F4','F7','F8','Fz']}
TEMPORAL_CH = {'CHBMIT':['T7-P7','T7-FT9','FT10-T8','T8-P8-0'],
               'Siena':['T3','T4','T5','T6'],
               'Mendeley':['T3','T4','T5','T6']}
CENTRAL_CH  = {'CHBMIT':['F3-C3','F4-C4'],'Siena':['C3','C4'],'Mendeley':['C3','C4']}
PARIETAL_CH = {'CHBMIT':['P3-O1','P4-O2'],'Siena':['P3','P4'],'Mendeley':['P3','P4']}
OCCIPITAL_CH= {'CHBMIT':['P7-O1','P8-O2'],'Siena':['O1','O2'],'Mendeley':['O1','O2']}
R0_CORE_CH  = {'CHBMIT':['T7-FT9','FT10-T8'],'Siena':['T3','T4'],'Mendeley':['T3','T4']}

REGION_DS = ['CHBMIT','Siena','Mendeley']
LEVEL_CHANNELS = {'R5':{},'R3':{},'R2':{},'R1':{},'R0':{}}
for _ds in REGION_DS:
    LEVEL_CHANNELS['R5'][_ds] = FRONTAL_CH[_ds]+TEMPORAL_CH[_ds]+CENTRAL_CH[_ds]+PARIETAL_CH[_ds]+OCCIPITAL_CH[_ds]
    LEVEL_CHANNELS['R3'][_ds] = FRONTAL_CH[_ds]+TEMPORAL_CH[_ds]+CENTRAL_CH[_ds]
    LEVEL_CHANNELS['R2'][_ds] = FRONTAL_CH[_ds]+TEMPORAL_CH[_ds]
    LEVEL_CHANNELS['R1'][_ds] = TEMPORAL_CH[_ds]
    LEVEL_CHANNELS['R0'][_ds] = R0_CORE_CH[_ds]

CH_RENAME = {'SeizeIT2': {'bteleftsd':'t3','bterightsd':'t4','crosstopsd':'t3'}}
SEIZEIT2_R0_TARGETS = ['t3','t4']

# Validações
_EXPECTED = {'R5':17,'R3':13,'R2':11,'R1':4,'R0':2}
_ORDER    = ['R0','R1','R2','R3','R5']
for _lv, _exp in _EXPECTED.items():
    for _ds in REGION_DS:
        _c = LEVEL_CHANNELS[_lv][_ds]
        assert len(_c)==_exp, f'{_lv}/{_ds}: {len(_c)} != {_exp}'
        assert len(set(_c))==len(_c), f'{_lv}/{_ds}: duplicado'
for _ds in REGION_DS:
    for _i in range(len(_ORDER)-1):
        assert set(LEVEL_CHANNELS[_ORDER[_i]][_ds]) <= set(LEVEL_CHANNELS[_ORDER[_i+1]][_ds])
del _lv,_exp,_ds,_c,_i

# Cria diretórios de features
for lv in LEVELS:
    os.makedirs(os.path.join(FEAT_DIR, lv), exist_ok=True)

# ── Slices de colunas para derivar níveis a partir do R5 ──────────────────
# Calculados analiticamente: cada canal ocupa N_FEAT=19 colunas consecutivas
# no vetor de features do R5, na ordem de LEVEL_CHANNELS['R5'][ds].
# Verificado numericamente: todos os slices são contíguos nos 3 datasets.
#
# Offsets (iguais para CHBMIT, Siena e Mendeley por construção):
#   R5: [0:323]   (17 canais × 19)
#   R3: [0:247]   (13 canais × 19 — frontal+temporal+central são os primeiros 13)
#   R2: [0:209]   (11 canais × 19 — frontal+temporal são os primeiros 11)
#   R1: [133:209] (4 canais × 19  — temporal começa no canal 7, índice 7*19=133)
#   R0: varia por dataset:
#       CHBMIT:          [152:190]  (T7-FT9=canal8, FT10-T8=canal9 → 8*19=152)
#       Siena/Mendeley:  [133:171]  (T3=canal7, T4=canal8 → 7*19=133)
#
# Estes slices são pré-computados uma vez aqui e usados em process_file_r5.
N_FEAT = 19

def _compute_r5_slices(ds):
    '''Retorna dict {nível: slice} para fatiar o vetor R5 de um dataset.'''
    R5 = LEVEL_CHANNELS['R5'][ds]
    slices = {}
    for lv in ['R5','R3','R2','R1','R0']:
        ch_list = LEVEL_CHANNELS[lv][ds]
        idxs = [R5.index(ch) for ch in ch_list]
        col_start = idxs[0]  * N_FEAT
        col_end   = (idxs[-1]+1) * N_FEAT
        slices[lv] = slice(col_start, col_end)
    return slices

R5_SLICES = {ds: _compute_r5_slices(ds) for ds in REGION_DS}

# ── Parâmetros de janelamento ──
WIN_SEC    = 30
STEP_SEC   = 15
MIN_PURITY = 0.80
LBL        = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)

# ── Features ──
BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,40)}
FEAT_NAMES = ['std','var','rms','line_len','mobility','skewness','kurtosis',
              'delta','theta','alpha','beta','gamma','sp_entropy','beta_rel',
              'dwt_d1','dwt_d2','dwt_d3','dwt_d4','complexity']


def normalize_ch(name, rename_map=None):
    s = str(name).lower()
    for sub in ['eeg','-ref','-le','-a1','-a2','ref']: s = s.replace(sub,' ')
    s = s.strip().replace(' ','')
    if '-' in s: s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]','',s)
    return rename_map.get(s,s) if rename_map else s

def normalize_ch_full(name):
    s = str(name).lower()
    for sub in ['eeg','-ref','-le','-a1','-a2','ref']: s = s.replace(sub,' ')
    s = s.strip().replace(' ','')
    return re.sub(r'[^a-z0-9-]','',s)

def channel_feats(sig, sfreq=256.0):
    if len(sig) < 2: return np.zeros(N_FEAT, dtype=np.float32)
    d1 = np.diff(sig); d2 = np.diff(d1) if len(d1)>1 else np.array([0.0])
    act = float(np.var(sig)); mob = float(np.sqrt(np.var(d1)/(act+1e-10)))
    mob2 = float(np.sqrt(np.var(d2)/(np.var(d1)+1e-10)))
    temporal = [float(np.std(sig)), act, float(np.sqrt(np.mean(sig**2))),
                float(np.sum(np.abs(d1))), mob, float(skew(sig)), float(kurtosis(sig))]
    nperseg = min(int(sfreq), max(int(sfreq//2), len(sig)//2))
    nperseg = max(min(nperseg, len(sig)), 2)
    f, psd = welch(sig, fs=sfreq, nperseg=nperseg); total = psd.sum()+1e-10
    bp = []
    for lo,hi in BANDS.values():
        idx = (f>=lo)&(f<=hi)
        bp.append(float(np.trapz(psd[idx],f[idx])) if idx.sum()>1 else 0.0)
    pn = psd/total; pn = pn[pn>0]; sp_ent = float(-np.sum(pn*np.log(pn)))
    b_idx = (f>=13)&(f<=30)
    beta_rel = float(np.trapz(psd[b_idx],f[b_idx])/total) if b_idx.sum()>1 else 0.0
    spectral = bp + [sp_ent, beta_rel]
    ml = min(4, pywt.dwt_max_level(len(sig),'db4')) if len(sig)>1 else 0
    if ml>=1:
        coeffs = pywt.wavedec(sig if sig.dtype==np.float64 else sig.astype(np.float64),'db4',level=ml)
        dwt = [float(np.sum(c**2)) for c in coeffs[1:5]]
        while len(dwt)<4: dwt.append(0.0)
    else:
        dwt = [0.0]*4
    complexity = mob2/(mob+1e-10)
    return np.array(temporal+spectral+dwt+[complexity], dtype=np.float32)

def build_fvec_r5(win_data, ch_names_raw, dataset):
    '''Calcula features apenas para R5 (todos os canais).
    Os outros níveis são derivados por fatiamento — não chamam esta função.'''
    ch_map = {normalize_ch_full(c):i for i,c in enumerate(ch_names_raw)}
    parts = []
    for tgt in LEVEL_CHANNELS['R5'][dataset]:
        i = ch_map.get(normalize_ch_full(tgt))
        parts.append(channel_feats(win_data[i]) if i is not None
                     else np.zeros(N_FEAT, dtype=np.float32))
    return np.concatenate(parts)   # shape (323,)

def build_fvec_seizeit2(win_data, ch_names_raw):
    '''SeizeIT2: só R0, 2 canais behind-the-ear.'''
    rename = CH_RENAME.get('SeizeIT2',{})
    ch_map = {normalize_ch(c,rename):i for i,c in enumerate(ch_names_raw)}
    parts = []
    for tgt in SEIZEIT2_R0_TARGETS:
        i = ch_map.get(tgt)
        parts.append(channel_feats(win_data[i]) if i is not None
                     else np.zeros(N_FEAT, dtype=np.float32))
    return np.concatenate(parts)   # shape (38,)


def _save_level(out_path, Xp, Xi, tp, ti, ch_list, dataset, patient, fkey,
                level, window_min, file_order, n_seizures,
                col_start=0, col_end=None):
    '''Salva um arquivo de features para um nível específico.

    Metadados salvos no .npz:
      X_pre / X_inter  — matrizes de features (n_janelas × n_features)
      t_pre / t_inter  — timestamps em segundos desde o início do arquivo
      channels         — nomes dos canais neste nível (ex: ['T7-FT9', 'FT10-T8'])
      feat_names       — nomes das 19 features por canal
      dataset, patient, fkey, level — identificação
      window_min       — janela pré-ictal em minutos (10/15/30/45)
      file_order       — índice cronológico do arquivo dentro do paciente
      n_seizures       — número de crises neste arquivo
      n_pre, n_inter, n_total — contagem de janelas
      win_sec, step_sec — tamanho e passo da janela em segundos
      n_feat_per_ch    — número de features por canal (19)
      sfreq            — frequência de amostragem em Hz
      col_start/col_end — offsets de coluna no vetor R5 (para auditoria/rastreabilidade)
    '''
    if col_end is None:
        col_end = Xp.shape[1] if len(Xp) else Xi.shape[1] if len(Xi) else 0
    np.savez_compressed(
        out_path,
        X_pre=Xp,   t_pre=tp,
        X_inter=Xi, t_inter=ti,
        channels=np.array(ch_list),
        feat_names=np.array(FEAT_NAMES),
        dataset=np.str_(dataset), patient=np.str_(patient),
        fkey=np.str_(fkey), level=np.str_(level),
        window_min=np.int64(window_min),
        file_order=np.int64(file_order),
        n_seizures=np.int64(n_seizures),
        n_pre=np.int64(len(Xp)), n_inter=np.int64(len(Xi)),
        n_total=np.int64(len(Xp)+len(Xi)),
        win_sec=np.float32(WIN_SEC), step_sec=np.float32(STEP_SEC),
        n_feat_per_ch=np.int32(N_FEAT),
        col_start=np.int32(col_start),
        col_end=np.int32(col_end),
    )


def process_file_r5(dataset, patient, fkey, window_min, file_order):
    '''Processa UMA gravação de CHBMIT/Siena/Mendeley para UMA janela pré-ictal.
    Calcula features R5 uma única vez e deriva R3/R2/R1/R0 por fatiamento.
    Salva 5 arquivos .npz (um por nível) em uma única passagem pelo sinal.

    Retorna lista de dicts de metadados (um por nível salvo com sucesso).
    '''
    results = []

    # Verifica quais níveis já estão prontos para essa gravação+janela
    out_paths = {
        lv: os.path.join(FEAT_DIR, lv, f'{dataset}__{patient}__{fkey}__w{window_min}.npz')
        for lv in LEVELS
    }
    pending_levels = [lv for lv in LEVELS if not os.path.exists(out_paths[lv])]

    if not pending_levels:
        # Todos os 5 níveis já existem — retorna metadados do cache
        for lv, out_path in out_paths.items():
            try:
                z = np.load(out_path, allow_pickle=True)
                results.append(dict(
                    dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    n_pre=int(z['n_pre']), n_inter=int(z['n_inter']),
                    n_total=int(z['n_total']), n_seizures=int(z['n_seizures']),
                    status='cached', path=out_path))
                z.close()
            except Exception:
                pass
        return results

    # ── Carrega rótulos (usa w{window_min}) ──
    l1_path = os.path.join(L1_DIR, f'{dataset}__{patient}__{fkey}_L1_w{window_min}.npz')
    if not os.path.exists(l1_path):
        for lv in pending_levels:
            results.append(dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                                window_min=window_min, file_order=file_order,
                                status='sem_l1', path=None))
        return results

    try:
        z1     = np.load(l1_path, allow_pickle=True)
        labels = z1['labels']
        sfreq  = float(z1['sfreq'])
        chs    = list(z1['ch_names'])
        n      = int(z1['n_samples'])
        z1.close(); del z1
    except Exception as e:
        for lv in pending_levels:
            results.append(dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                                window_min=window_min, file_order=file_order,
                                status=f'erro_l1:{e}', path=None))
        return results

    # ── Carrega sinal bruto com mmap (leitura sob demanda, não carrega tudo na RAM) ──
    sig_path = os.path.join(SIGNAL_DIR, f'{dataset}__{patient}__{fkey}_signal.npz')
    if not os.path.exists(sig_path):
        del labels
        for lv in pending_levels:
            results.append(dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                                window_min=window_min, file_order=file_order,
                                status='sem_signal', path=None))
        return results

    try:
        zs   = np.load(sig_path, allow_pickle=True, mmap_mode='r')
        data = zs['data']   # array mapeado — páginas carregadas sob demanda
    except Exception as e:
        del labels
        for lv in pending_levels:
            results.append(dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                                window_min=window_min, file_order=file_order,
                                status=f'erro_signal:{e}', path=None))
        return results

    # ── Conta crises ──
    ictal_arr = (labels == LBL['ictal']).astype(int)
    n_seizures = int(np.sum(np.diff(np.concatenate([[0], ictal_arr])) == 1))

    # ── Extração de janelas ── (percorre o sinal UMA VEZ, calcula R5, salva tudo)
    win_n  = int(WIN_SEC * sfreq)
    step_n = max(1, int(STEP_SEC * sfreq))
    slices = R5_SLICES[dataset]   # dict {lv: slice} pré-calculado

    # Acumuladores por nível
    X_pre_r5  = []; X_inter_r5  = []
    t_pre     = []; t_inter     = []

    for start in range(0, n - win_n + 1, step_n):
        wl    = labels[start:start+win_n]
        valid = wl[wl >= 0]
        if len(valid) == 0:
            continue
        vals, counts = np.unique(valid, return_counts=True)
        dom = vals[np.argmax(counts)]
        if dom not in (LBL['interictal'], LBL['preictal']):
            continue
        if counts.max() / win_n < MIN_PURITY:
            continue

        # Calcula features R5 (17 canais × 19 = 323 features)
        fv_r5 = build_fvec_r5(data[:, start:start+win_n], chs, dataset)
        t_sec = start / sfreq

        if dom == LBL['preictal']:
            X_pre_r5.append(fv_r5); t_pre.append(t_sec)
        else:
            X_inter_r5.append(fv_r5); t_inter.append(t_sec)

    del data; gc.collect()

    # Converte para arrays numpy
    Xp5 = np.array(X_pre_r5,   dtype=np.float32) if X_pre_r5   else np.empty((0,323),dtype=np.float32)
    Xi5 = np.array(X_inter_r5, dtype=np.float32) if X_inter_r5 else np.empty((0,323),dtype=np.float32)
    tp  = np.array(t_pre,   dtype=np.float64)
    ti  = np.array(t_inter, dtype=np.float64)

    del X_pre_r5, X_inter_r5, labels; gc.collect()

    n_total = len(Xp5) + len(Xi5)

    # ── Salva cada nível pendente por fatiamento ──
    for lv in pending_levels:
        sl = slices[lv]
        Xp_lv = Xp5[:, sl] if len(Xp5) else np.empty((0, sl.stop-sl.start), dtype=np.float32)
        Xi_lv = Xi5[:, sl] if len(Xi5) else np.empty((0, sl.stop-sl.start), dtype=np.float32)

        if n_total == 0:
            results.append(dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                                window_min=window_min, file_order=file_order,
                                n_pre=0, n_inter=0, n_total=0, n_seizures=n_seizures,
                                status='vazio', path=None))
            continue

        ch_list = LEVEL_CHANNELS[lv][dataset]
        _save_level(out_paths[lv], Xp_lv, Xi_lv, tp, ti,
                    ch_list, dataset, patient, fkey, lv,
                    window_min, file_order, n_seizures,
                    col_start=sl.start, col_end=sl.stop)

        results.append(dict(
            dataset=dataset, patient=patient, fkey=fkey, level=lv,
            window_min=window_min, file_order=file_order,
            n_pre=len(Xp_lv), n_inter=len(Xi_lv),
            n_total=len(Xp_lv)+len(Xi_lv),
            n_seizures=n_seizures, status='ok', path=out_paths[lv]))

    del Xp5, Xi5, tp, ti; gc.collect()
    return results


def process_file_seizeit2(patient, fkey, window_min, file_order):
    '''SeizeIT2: processa só R0, canais behind-the-ear (t3, t4).
    Fluxo separado porque os canais são completamente diferentes de R5.
    '''
    dataset = 'SeizeIT2'
    lv      = 'R0'
    out_path = os.path.join(FEAT_DIR, lv, f'{dataset}__{patient}__{fkey}__w{window_min}.npz')

    if os.path.exists(out_path):
        try:
            z = np.load(out_path, allow_pickle=True)
            r = dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                     window_min=window_min, file_order=file_order,
                     n_pre=int(z['n_pre']), n_inter=int(z['n_inter']),
                     n_total=int(z['n_total']), n_seizures=int(z['n_seizures']),
                     status='cached', path=out_path)
            z.close()
            return r
        except Exception:
            pass

    l1_path = os.path.join(L1_DIR, f'{dataset}__{patient}__{fkey}_L1_w{window_min}.npz')
    if not os.path.exists(l1_path):
        return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    status='sem_l1', path=None)
    try:
        z1 = np.load(l1_path, allow_pickle=True)
        labels = z1['labels']; sfreq = float(z1['sfreq'])
        chs = list(z1['ch_names']); n = int(z1['n_samples'])
        z1.close(); del z1
    except Exception as e:
        return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    status=f'erro_l1:{e}', path=None)

    sig_path = os.path.join(SIGNAL_DIR, f'{dataset}__{patient}__{fkey}_signal.npz')
    if not os.path.exists(sig_path):
        del labels
        return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    status='sem_signal', path=None)
    try:
        zs = np.load(sig_path, allow_pickle=True, mmap_mode='r')
        data = zs['data']
    except Exception as e:
        del labels
        return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    status=f'erro_signal:{e}', path=None)

    ictal_arr = (labels == LBL['ictal']).astype(int)
    n_seizures = int(np.sum(np.diff(np.concatenate([[0], ictal_arr])) == 1))
    win_n  = int(WIN_SEC * sfreq)
    step_n = max(1, int(STEP_SEC * sfreq))
    X_pre = []; X_inter = []; t_pre = []; t_inter = []

    for start in range(0, n - win_n + 1, step_n):
        wl    = labels[start:start+win_n]
        valid = wl[wl >= 0]
        if len(valid) == 0: continue
        vals, counts = np.unique(valid, return_counts=True)
        dom = vals[np.argmax(counts)]
        if dom not in (LBL['interictal'], LBL['preictal']): continue
        if counts.max() / win_n < MIN_PURITY: continue
        fv = build_fvec_seizeit2(data[:, start:start+win_n], chs)
        t  = start / sfreq
        if dom == LBL['preictal']: X_pre.append(fv); t_pre.append(t)
        else:                        X_inter.append(fv); t_inter.append(t)

    del data, labels; gc.collect()
    Xp = np.array(X_pre,   dtype=np.float32) if X_pre   else np.empty((0,38),dtype=np.float32)
    Xi = np.array(X_inter, dtype=np.float32) if X_inter else np.empty((0,38),dtype=np.float32)
    tp = np.array(t_pre,   dtype=np.float64)
    ti = np.array(t_inter, dtype=np.float64)

    if len(Xp)+len(Xi) == 0:
        return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                    window_min=window_min, file_order=file_order,
                    n_pre=0, n_inter=0, n_total=0, n_seizures=n_seizures,
                    status='vazio', path=None)

    _save_level(out_path, Xp, Xi, tp, ti, SEIZEIT2_R0_TARGETS,
                dataset, patient, fkey, lv, window_min, file_order, n_seizures,
                col_start=0, col_end=38)
    return dict(dataset=dataset, patient=patient, fkey=fkey, level=lv,
                window_min=window_min, file_order=file_order,
                n_pre=len(Xp), n_inter=len(Xi), n_total=len(Xp)+len(Xi),
                n_seizures=n_seizures, status='ok', path=out_path)


def build_task_list_v5():
    '''Monta lista de tarefas.
    - CHBMIT/Siena/Mendeley: 1 tarefa por (fkey, window_min) — processa todos os 5 níveis
    - SeizeIT2: 1 tarefa por (fkey, window_min) — só R0

    Uma tarefa é "pendente" se QUALQUER nível esperado ainda não existe no disco.
    Isso garante retomada correta mesmo se apenas alguns níveis foram salvos.
    '''
    tasks_r5  = []   # (dataset, patient, fkey, window_min, file_order)
    tasks_s2  = []   # (patient,  fkey, window_min, file_order)

    for ds in ['CHBMIT','Siena','Mendeley']:
        for pat in PATIENTS[ds]:
            ref_files = sorted(glob.glob(os.path.join(L1_DIR, f'{ds}__{pat}__*_L1_w{PREICTAL_WINDOWS_MIN[0]}.npz')))
            if not ref_files:
                ref_files_any = sorted(glob.glob(os.path.join(L1_DIR, f'{ds}__{pat}__*_L1_w*.npz')))
                seen = set(); unique = []
                for fp in ref_files_any:
                    fk = os.path.basename(fp).replace(f'{ds}__{pat}__','').split('_L1_')[0]
                    if fk not in seen: seen.add(fk); unique.append(fp)
                ref_files = unique

            for file_order, ref_fp in enumerate(ref_files):
                bn   = os.path.basename(ref_fp)
                fkey = bn.replace(f'{ds}__{pat}__','').split('_L1_')[0]
                for w in PREICTAL_WINDOWS_MIN:
                    # Pendente se QUALQUER dos 5 níveis não existe
                    any_missing = any(
                        not os.path.exists(os.path.join(FEAT_DIR, lv, f'{ds}__{pat}__{fkey}__w{w}.npz'))
                        for lv in LEVELS
                    )
                    if any_missing:
                        tasks_r5.append((ds, pat, fkey, w, file_order))

    for pat in PATIENTS['SeizeIT2']:
        ref_files = sorted(glob.glob(os.path.join(L1_DIR, f'SeizeIT2__{pat}__*_L1_w{PREICTAL_WINDOWS_MIN[0]}.npz')))
        if not ref_files:
            ref_files_any = sorted(glob.glob(os.path.join(L1_DIR, f'SeizeIT2__{pat}__*_L1_w*.npz')))
            seen = set(); unique = []
            for fp in ref_files_any:
                fk = os.path.basename(fp).replace(f'SeizeIT2__{pat}__','').split('_L1_')[0]
                if fk not in seen: seen.add(fk); unique.append(fp)
            ref_files = unique

        for file_order, ref_fp in enumerate(ref_files):
            bn   = os.path.basename(ref_fp)
            fkey = bn.replace(f'SeizeIT2__{pat}__','').split('_L1_')[0]
            for w in PREICTAL_WINDOWS_MIN:
                out = os.path.join(FEAT_DIR, 'R0', f'SeizeIT2__{pat}__{fkey}__w{w}.npz')
                if not os.path.exists(out):
                    tasks_s2.append((pat, fkey, w, file_order))

    return tasks_r5, tasks_s2


Overwriting nb3_worker.py


## 3. Importa o módulo e configura paralelismo

### Escolha de `N_JOBS`

Esta máquina tem **4 núcleos físicos** com hyperthreading (8 lógicos).
O NB3 é **CPU-bound puro** — Welch e DWT são operações de ponto flutuante intensas
que saturam completamente um núcleo físico. Por isso:

- `N_JOBS = 4` → ocupa os 4 núcleos físicos completamente; os 4 threads lógicos
  extras ficam livres para Windows, Jupyter e I/O de disco. **Recomendado.**
- `N_JOBS = 3` → deixa 1 núcleo físico livre; use se quiser usar o PC normalmente
  enquanto a extração roda (navegar, ver vídeo, etc.).
- `N_JOBS = 6-7` → **não recomendado** para cálculo numérico pesado: threads lógicos
  do mesmo núcleo físico compartilham FPU e cache L1/L2, causando contenção sem ganho
  real de velocidade, mas travando o PC.

### Retomada automática

Após rodar a célula de importação (`build_task_list_v5()` é chamado aqui para exibir
o progresso atual), você verá quantas tarefas ainda estão pendentes. Se interrompeu
a execução anterior, basta executar as células 3 e 5 (importação + execução) para continuar.


In [3]:
import importlib
import nb3_worker
importlib.reload(nb3_worker)
from nb3_worker import (
    process_file_r5, process_file_seizeit2, build_task_list_v5,
    PATIENTS, PREICTAL_WINDOWS_MIN, LEVELS, LEVEL_DS, LEVEL_CHANNELS,
    FEAT_DIR, L1_DIR, LOG_DIR, N_FEAT, FEAT_NAMES,
)

import glob, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import IPython.display as ipd

# ── Paralelismo ──────────────────────────────────────────────────────────
# Quadcore com hyperthreading (8 lógicos): N_JOBS = 4 é o ideal.
# Cada worker ocupa 1 núcleo físico. Com 4 workers os 4 físicos ficam
# saturados e os 4 threads lógicos extras ficam livres para Windows/Jupyter.
# Se quiser usar o PC enquanto roda, reduza para 3.
N_JOBS = 4

print('Configuração carregada.')
print(f'  Pacientes: {sum(len(v) for v in PATIENTS.values())} '
      f'({", ".join(f"{k}:{len(v)}" for k,v in PATIENTS.items())})')
print(f'  Níveis: {LEVELS}')
print(f'  Janelas: {PREICTAL_WINDOWS_MIN} min')
print(f'  Workers: {N_JOBS}')
print()
print('Arquitetura v5:')
print('  CHBMIT/Siena/Mendeley: 1 passagem pelo sinal → 5 níveis por fatiamento')
print('  SeizeIT2: R0 direto (2 canais behind-the-ear)')
tasks_r5, tasks_s2 = build_task_list_v5()
print(f'  Tarefas pendentes: {len(tasks_r5)} gravações×janelas R5 + {len(tasks_s2)} SeizeIT2')


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuração carregada.
  Pacientes: 42 (CHBMIT:12, Siena:12, Mendeley:6, SeizeIT2:12)
  Níveis: ['R5', 'R3', 'R2', 'R1', 'R0']
  Janelas: [10, 15, 30, 45] min
  Workers: 4

Arquitetura v5:
  CHBMIT/Siena/Mendeley: 1 passagem pelo sinal → 5 níveis por fatiamento
  SeizeIT2: R0 direto (2 canais behind-the-ear)
  Tarefas pendentes: 632 gravações×janelas R5 + 488 SeizeIT2


## 4. Função de extração paralela

### Como funciona

`run_extraction()` orquestra toda a extração:

1. Chama `build_task_list_v5()` para listar tarefas pendentes
2. Exibe progresso atual (quantas gravações×janelas já foram concluídas)
3. Submete tarefas em **lotes** (`CHUNK = n_jobs × 4`) ao `ProcessPoolExecutor`
4. Para cada tarefa concluída: contabiliza resultado e atualiza a barra de progresso
5. Em caso de erro transitório (`erro_l1`, `erro_signal`): **retenta até 3 vezes**
   com espera crescente (0.5s, 1.0s, 1.5s)
6. Ao final: exibe resumo completo

### Tipos de resultado por tarefa

| Status | Significado | Retry? |
|--------|-------------|--------|
| `ok` | Features extraídas e salvas com sucesso | — |
| `cached` | Arquivo já existia em disco, pulado | — |
| `vazio` | Gravação sem janelas válidas (sem crise ou muito curta) | Não |
| `sem_l1` | Arquivo de rótulos do NB2 não encontrado | Não |
| `sem_signal` | Arquivo de sinal do NB1 não encontrado | Não |
| `erro_l1` | Falha ao ler rótulos (possível corrompimento) | **Sim (3×)** |
| `erro_signal` | Falha ao ler sinal (possível corrompimento) | **Sim (3×)** |

### Submissão em lotes

Em vez de submeter todas as ~1000 tarefas de uma vez (que inflaria a RAM com um dicionário
gigante de `futures`), o código submete `CHUNK` tarefas por vez e espera pela conclusão
antes de submeter o próximo lote. Isso mantém a RAM sob controle sem perder paralelismo.

### `max_tasks_per_child=50`

No Windows, o método `spawn` cria um processo filho do zero para cada worker. Com o tempo,
processos acumulam fragmentação de memória. `max_tasks_per_child=50` recicla cada processo
após 50 tarefas — sem isso o consumo de RAM cresce continuamente. O valor 50 é um equilíbrio:
mais alto que o padrão (que era 20, custando muitos reinicios) mas baixo o suficiente para
controlar a memória.


In [4]:
def run_extraction(n_jobs=None):
    '''Extrai features com a arquitetura v5:
    - CHBMIT/Siena/Mendeley: calcula R5 uma vez, deriva R3/R2/R1/R0 por fatiamento
    - SeizeIT2: processa R0 diretamente (canais behind-the-ear)
    '''
    if n_jobs is None:
        n_jobs = N_JOBS

    tasks_r5, tasks_s2 = build_task_list_v5()
    total_pendentes = len(tasks_r5) + len(tasks_s2)

    if total_pendentes == 0:
        print('Nada pendente — tudo já foi processado.')
        return []

    # Conta arquivos já em disco — fonte de verdade real, não depende dos L1 existirem.
    # Evita o bug de contador negativo quando o L1 da janela de referência ainda não existe.
    ja_feitos = sum(
        1 for lv in LEVELS
        for _ in glob.glob(os.path.join(FEAT_DIR, lv, '*__w*.npz'))
    )
    total_arquivos_esperados = ja_feitos + len(tasks_r5) * 5 + len(tasks_s2)

    print(f"Tarefas pendentes: {len(tasks_r5)} R5+derivados + {len(tasks_s2)} SeizeIT2")
    print(f"Arquivos .npz em disco:  {ja_feitos} | Pendentes: {len(tasks_r5)*5 + len(tasks_s2)} | Total ao final: {total_arquivos_esperados}")
    print(f"Paralelismo: {n_jobs} workers (cada tarefa = 1 gravação × 1 janela → 5 níveis)")
    t_est = total_pendentes * 15 / (n_jobs * 3600)
    print(f"Estimativa: ~{t_est:.1f}h")

    from concurrent.futures import ProcessPoolExecutor, as_completed
    import time

    # Barra de progresso em tarefas (cada tarefa = 1 gravação × 1 janela)
    pbar = tqdm(
        total=total_pendentes,
        desc='Extração',
        unit='tarefa',
        ncols=120,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}'
    )

    results = []
    ok_count = warn_count = err_count = 0
    t_start = time.time()
    all_tasks = [('r5', t) for t in tasks_r5] + [('s2', t) for t in tasks_s2]
    CHUNK = n_jobs * 4

    with ProcessPoolExecutor(max_workers=n_jobs, max_tasks_per_child=50) as executor:
        for chunk_start in range(0, len(all_tasks), CHUNK):
            chunk = all_tasks[chunk_start:chunk_start+CHUNK]
            futures = {}
            for kind, task in chunk:
                if kind == 'r5':
                    ds, pat, fkey, w, fo = task
                    f = executor.submit(process_file_r5, ds, pat, fkey, w, fo)
                    futures[f] = ('r5', ds, pat, fkey, w, fo)
                else:
                    pat, fkey, w, fo = task
                    f = executor.submit(process_file_seizeit2, pat, fkey, w, fo)
                    futures[f] = ('s2', 'SeizeIT2', pat, fkey, w, fo)

            for future in as_completed(futures):
                kind, ds, pat, fkey, w, fo = futures[future]
                try:
                    r = future.result()
                    if isinstance(r, list):
                        results.extend(r)
                        n_ok   = sum(1 for x in r if x and x.get('status') == 'ok')
                        n_warn = sum(1 for x in r if x and x.get('status','').startswith(('erro','sem','vazio')))
                        ok_count += n_ok; warn_count += n_warn
                    elif r:
                        results.append(r)
                        if r.get('status') == 'ok': ok_count += 1
                        elif r.get('status','').startswith(('erro','sem','vazio')): warn_count += 1
                except Exception as e:
                    err_count += 1
                    results.append(None)
                    r = None

                # ── Barra de progresso rica ─────────────────────────────────
                elapsed = time.time() - t_start
                done = pbar.n + 1
                rate = done / elapsed if elapsed > 0 else 0

                # Status da última tarefa concluída
                if isinstance(r, list) and r:
                    status_info = f"{len(r)} níveis ok" if any(x and x.get('status')== 'ok' for x in r) else 'vazio/erro'
                    n_pre_total = sum(x.get('n_pre',0) for x in r if x)
                elif r and isinstance(r, dict):
                    status_info = r.get('status','?')
                    n_pre_total = r.get('n_pre', 0)
                else:
                    status_info = 'erro'; n_pre_total = 0

                postfix_str = (
                    f"ds={ds} pat={pat} "
                    f"fkey={fkey[-16:] if len(fkey)>16 else fkey} "  # trunca fkeys longos
                    f"w={w}min "
                    f"ok={ok_count} warn={warn_count+err_count} "
                    f"pre={n_pre_total}jan"
                )
                pbar.set_postfix_str(postfix_str, refresh=True)
                pbar.update(1)

    pbar.close()

    elapsed_total = time.time() - t_start
    print(f'\n{"="*60}')
    print(f'Extração concluída em {elapsed_total/3600:.2f}h ({elapsed_total:.0f}s)')
    print(f'  Tarefas processadas: {total_pendentes}')
    print(f'  Arquivos .npz novos: {ok_count}')
    print(f'  Vazios (sem janelas): {warn_count}')
    print(f'  Erros:               {err_count}')
    print(f'{"="*60}')

    erros = [r for r in results if r and isinstance(r, dict)
             and r.get('status','').startswith(('erro','sem'))]
    if erros:
        print(f'\nDetalhes dos erros ({len(erros)} total, primeiros 20):')
        for e in erros[:20]:
            print(f'  {e.get("dataset","?")}/{e.get("patient","?")}/{e.get("fkey","?")}/'
                  f'L{e.get("level","?")} w{e.get("window_min","?")}: {e.get("status","?")}')
    return results


print('run_extraction definido (v5).')


run_extraction definido (v5).


## 5. Execução

**Pode interromper e reexecutar a qualquer momento.** A retomada é automática:
`build_task_list_v5()` verifica quais (gravação × janela) ainda têm algum nível
faltando e só processa esses. Gravações com todos os 5 níveis completos são ignoradas.

Para testar antes de rodar tudo:
```python
run_extraction(n_jobs=2)                    # menos workers, PC mais responsivo
```

**Leitura da barra de progresso:**
```
Extração |████░░| 234/1016 [23:14<1:45:00, 2.1tarefa/s] ds=CHBMIT pat=chb04 fkey=chb04_07 w=30min ok=1156 err=0 warn=2 pre=18jan
```
- `ds/pat/fkey` — qual gravação está sendo processada agora
- `w=Xmin` — qual janela pré-ictal
- `ok` — arquivos `.npz` novos gerados com sucesso (acumulado)
- `err` — erros após todas as tentativas de retry (acumulado)
- `warn` — gravações sem dado (sem_l1/sem_signal/vazio, acumulado)
- `pre=Xjan` — janelas pré-ictais extraídas na última tarefa concluída
- `retry=N` — aparece apenas quando houve retry na última tarefa


In [ ]:
results = run_extraction()


Tarefas pendentes: 632 R5+derivados + 488 SeizeIT2
Arquivos .npz em disco:  1440 | Pendentes: 3648 | Total ao final: 5088
Paralelismo: 4 workers (cada tarefa = 1 gravação × 1 janela → 5 níveis)
Estimativa: ~1.2h


Extração:  54%|▌| 604/1120 [44:06<54:17,  6.31s/tarefa] , ds=Mendeley pat=p13 fkey=p13_Record4 w=45min ok=3020 warn=0 pr

## 6. Manifesto global

Gera `data/logs/nb3_manifest.csv` com **uma linha por arquivo `.npz` gerado**.
O NB4 usa esse CSV para listar gravações disponíveis de forma eficiente — sem fazer
`glob` em milhares de arquivos a cada execução.

**Pode ser executado a qualquer momento**, inclusive com extração parcial. O manifesto
refletirá exatamente o que está em disco no momento da execução.

Colunas do CSV: `dataset`, `patient`, `fkey`, `level`, `window_min`, `file_order`,
`n_seizures`, `n_pre`, `n_inter`, `n_total`, `path`.

O campo `file_order` é essencial para o NB4: permite ordenar cronologicamente as
gravações de cada paciente sem precisar abrir os EDFs originais.


In [ ]:
def build_manifest():
    '''Varre todos os arquivos de features gerados e constrói o manifesto.
    Pode ser chamada a qualquer momento, inclusive após extração parcial.'''
    rows = []
    pattern = os.path.join(FEAT_DIR, '*', f'*__w*.npz')
    all_files = sorted(glob.glob(pattern))
    print(f'Lendo metadados de {len(all_files)} arquivos...')
    for fp in all_files:
        try:
            z = np.load(fp, allow_pickle=True)
            rows.append(dict(
                dataset    = str(z['dataset']),
                patient    = str(z['patient']),
                fkey       = str(z['fkey']),
                level      = str(z['level']),
                window_min = int(z['window_min']),
                file_order = int(z['file_order']),
                n_seizures = int(z['n_seizures']),
                n_pre      = int(z['n_pre']),
                n_inter    = int(z['n_inter']),
                n_total    = int(z['n_total']),
                path       = fp,
            ))
            z.close()
        except Exception as e:
            rows.append(dict(path=fp, status=f'erro:{e}'))

    df = pd.DataFrame(rows)
    manifest_path = os.path.join(LOG_DIR, 'nb3_manifest.csv')
    df.to_csv(manifest_path, index=False)
    print(f'Manifesto salvo: {manifest_path} ({len(df)} linhas)')

    # Resumo
    if 'dataset' in df.columns:
        ok = df[df.get('n_total',pd.Series([0]*len(df))) > 0] if 'n_total' in df.columns else df
        print()
        print('Arquivos gerados por dataset e nível:')
        if 'level' in df.columns and 'dataset' in df.columns:
            ipd.display(df.groupby(['dataset','level']).size().unstack(fill_value=0))
        print()
        print('Gravações com pelo menos 1 janela pré-ictal (por dataset):')
        if 'n_pre' in df.columns:
            ipd.display(df[df['n_pre']>0].groupby('dataset').size().rename('gravações_com_preictal'))
    return df

df_manifest = build_manifest()


## 7. Verificação rápida (amostra de 20 arquivos)

Verifica se uma amostra aleatória de 20 arquivos pode ser carregada e tem a
**shape correta** (`n_features = n_canais × 19` para o dataset/nível do arquivo).

Útil para verificar rapidamente se a extração está funcionando durante a execução.
Para uma auditoria completa de todos os arquivos, use a **Seção 8** abaixo.


In [ ]:
# Verifica que os arquivos podem ser carregados e têm a shape certa
print('Verificando integridade de uma amostra de arquivos...')
all_feat_files = sorted(glob.glob(os.path.join(FEAT_DIR, '*', '*__w*.npz')))
sample = random.sample(all_feat_files, min(20, len(all_feat_files)))
erros_verif = []
for fp in sample:
    try:
        z = np.load(fp, allow_pickle=True)
        lv  = str(z['level']); ds = str(z['dataset'])
        Xp  = z['X_pre']; Xi = z['X_inter']
        n_ch = 2 if ds=='SeizeIT2' else len(LEVEL_CHANNELS[lv][ds])
        expected = n_ch * N_FEAT
        assert Xp.shape[1] == expected or len(Xp)==0, f'X_pre shape {Xp.shape} esperado ({len(Xp)},{expected})'
        assert Xi.shape[1] == expected or len(Xi)==0, f'X_inter shape {Xi.shape}'
        z.close()
    except Exception as e:
        erros_verif.append((fp, str(e)))

if erros_verif:
    print(f'ERROS em {len(erros_verif)} arquivos:')
    for fp, e in erros_verif:
        print(f'  {os.path.basename(fp)}: {e}')
else:
    print(f'OK — {len(sample)} arquivos verificados sem erro.')


## 8. Auditoria completa de integridade

Varre **todos** os arquivos gerados e verifica 4 dimensões:

### 1. Shape
Cada arquivo deve ter `n_features = n_canais_do_nível × 19`. Por exemplo:
- `CHBMIT/R5`: 17 canais × 19 = 323 colunas em `X_pre` e `X_inter`
- `Siena/R1`: 4 canais × 19 = 76 colunas
- `SeizeIT2/R0`: 2 canais × 19 = 38 colunas

### 2. NaN/Inf
Verifica ausência de valores inválidos em `X_pre` e `X_inter`. NaN pode aparecer
se um canal estava zerado ou ausente no EDF original.

### 3. Metadados
Confirma que `dataset`, `patient`, `fkey`, `level` salvos dentro do `.npz` batem
com o nome do arquivo e a pasta onde está. Um mismatch indicaria salvamento no
caminho errado.

### 4. Cobertura
Lista quais pacientes esperados (conforme `PATIENTS` no worker) foram encontrados
por dataset/nível, e quais estão faltando. Também mostra o total de janelas
pré-ictais extraídas por dataset/nível — útil para identificar pacientes com
poucos dados.

**Pode ser executada com extração parcial** — mostrará o estado atual do disco.


In [ ]:
# ── Verificação completa de integridade ──────────────────────────────────────
# Vai além da amostra de 20: verifica TODOS os arquivos gerados em 4 dimensões:
#   1. Shape correta (n_features = n_canais × 19)
#   2. Ausência de NaN/Inf em X_pre e X_inter
#   3. Metadados consistentes (dataset/patient/level batem com o nome do arquivo)
#   4. Cobertura: todos os pacientes/níveis/janelas esperados foram gerados
# ─────────────────────────────────────────────────────────────────────────────

import os, glob, re
import numpy as np
import pandas as pd
import IPython.display as ipd

FEAT_DIR_CHECK = FEAT_DIR   # data/features

print("=" * 70)
print(f"AUDITORIA DE INTEGRIDADE — {FEAT_DIR_CHECK}")
print("=" * 70)

all_files = sorted(glob.glob(os.path.join(FEAT_DIR_CHECK, '*', '*__w*.npz')))
print(f"\nTotal de arquivos encontrados: {len(all_files)}")
print(f"Total esperado:               {5 * 4 * 254}  (5 níveis × 4 janelas × ~254 gravações)")
print()

erros_shape    = []
erros_nan      = []
erros_meta     = []
ok_count       = 0
stats_rows     = []

for fp in all_files:
    bn = os.path.basename(fp)
    lv_dir = os.path.basename(os.path.dirname(fp))
    try:
        z = np.load(fp, allow_pickle=True)
        ds    = str(z['dataset'])
        pat   = str(z['patient'])
        fkey  = str(z['fkey'])
        lv    = str(z['level'])
        w     = int(z['window_min'])
        Xp    = z['X_pre']
        Xi    = z['X_inter']
        n_pre = int(z['n_pre'])
        n_int = int(z['n_inter'])
        z.close()

        # 1 — Shape
        n_ch = 2 if ds == 'SeizeIT2' else len(LEVEL_CHANNELS[lv][ds])
        expected_feats = n_ch * N_FEAT
        shape_ok = True
        if len(Xp) > 0 and Xp.shape[1] != expected_feats:
            erros_shape.append((bn, f'X_pre shape {Xp.shape}, esperado cols={expected_feats}'))
            shape_ok = False
        if len(Xi) > 0 and Xi.shape[1] != expected_feats:
            erros_shape.append((bn, f'X_inter shape {Xi.shape}, esperado cols={expected_feats}'))
            shape_ok = False

        # 2 — NaN/Inf
        nan_ok = True
        if len(Xp) > 0 and not np.isfinite(Xp).all():
            erros_nan.append((bn, 'X_pre contém NaN/Inf'))
            nan_ok = False
        if len(Xi) > 0 and not np.isfinite(Xi).all():
            erros_nan.append((bn, 'X_inter contém NaN/Inf'))
            nan_ok = False

        # 3 — Metadados consistentes com nome do arquivo
        # Padrão: {dataset}__{patient}__{fkey}__w{window_min}.npz
        expected_bn = f'{ds}__{pat}__{fkey}__w{w}.npz'
        meta_ok = (bn == expected_bn) and (lv == lv_dir)
        if not meta_ok:
            erros_meta.append((bn, f'esperado={expected_bn}, nível arquivo={lv_dir} vs meta={lv}'))

        if shape_ok and nan_ok and meta_ok:
            ok_count += 1

        stats_rows.append(dict(
            dataset=ds, level=lv, window_min=w,
            n_pre=n_pre, n_inter=n_int,
            ok=shape_ok and nan_ok and meta_ok
        ))

    except Exception as e:
        erros_shape.append((bn, f'ERRO AO LER: {e}'))

# ── Resumo de erros ──
print(f"Arquivos OK:             {ok_count}/{len(all_files)} ({ok_count/max(len(all_files),1)*100:.1f}%)")
print(f"Erros de shape:          {len(erros_shape)}")
print(f"Erros de NaN/Inf:        {len(erros_nan)}")
print(f"Erros de metadados:      {len(erros_meta)}")

if erros_shape:
    print("\n⚠ ERROS DE SHAPE (primeiros 10):")
    for fn, msg in erros_shape[:10]:
        print(f"  {fn}: {msg}")

if erros_nan:
    print("\n⚠ ERROS NaN/Inf (primeiros 10):")
    for fn, msg in erros_nan[:10]:
        print(f"  {fn}: {msg}")

if erros_meta:
    print("\n⚠ ERROS DE METADADOS (primeiros 10):")
    for fn, msg in erros_meta[:10]:
        print(f"  {fn}: {msg}")

# ── Cobertura por dataset / nível / janela ──
if stats_rows:
    df = pd.DataFrame(stats_rows)
    print("\n── Janelas pré-ictais com pelo menos 1 janela pré-ictal (n_pre > 0) ──")
    cov = df[df['n_pre'] > 0].groupby(['dataset','level','window_min']).size().unstack('window_min', fill_value=0)
    ipd.display(cov)

    print("\n── Total de janelas pré-ictais extraídas por dataset/nível ──")
    tot = df.groupby(['dataset','level'])['n_pre'].sum().unstack('level', fill_value=0)
    ipd.display(tot)

    print("\n── Arquivos por dataset/nível (esperado vs gerado) ──")
    cnt = df.groupby(['dataset','level']).size().unstack('level', fill_value=0)
    ipd.display(cnt)

# ── Cobertura de pacientes esperados ──
print("\n── Pacientes esperados vs encontrados ──")
found_pats = df.groupby(['dataset','level'])['patient'].apply(lambda x: set(x.unique())).reset_index() if stats_rows else pd.DataFrame()
for ds, pats_expected in PATIENTS.items():
    for lv in LEVELS:
        if ds not in LEVEL_DS.get(lv, []):
            continue
        if not stats_rows:
            print(f"  {ds}/{lv}: sem dados")
            continue
        sub = df[(df['dataset']==ds) & (df['level']==lv)]['patient'].unique()
        missing = set(pats_expected) - set(sub)
        extra   = set(sub) - set(pats_expected)
        status  = "OK" if not missing and not extra else "⚠"
        print(f"  {status} {ds}/{lv}: {len(sub)}/{len(pats_expected)} pacientes", end="")
        if missing: print(f"  | faltando: {sorted(missing)}", end="")
        if extra:   print(f"  | extras: {sorted(extra)}", end="")
        print()

print("\n" + "=" * 70)
if not erros_shape and not erros_nan and not erros_meta:
    print("✓ AUDITORIA CONCLUÍDA SEM ERROS")
else:
    print(f"✗ AUDITORIA CONCLUÍDA COM {len(erros_shape)+len(erros_nan)+len(erros_meta)} PROBLEMAS — verifique acima")
print("=" * 70)


## 9. Verificação cruzada R5 → níveis derivados

### O que verifica

Confirma que os arquivos de R3/R2/R1/R0 salvos em disco são **literalmente** as
colunas corretas do R5 correspondente — ou seja, que o fatiamento foi aplicado
corretamente e os dados são consistentes entre níveis.

Por exemplo, para `CHBMIT__chb01__chb01_03__w30.npz`:
- `R5/...npz` deve ter 323 colunas (17 canais × 19)
- `R1/...npz` deve ser **exatamente** `R5[:,133:209]` (canais temporais)
- `R0/...npz` deve ser **exatamente** `R5[:,152:190]` (T7-FT9 e FT10-T8)

### Por que é independente

Esta célula **não importa nem usa nenhuma variável do worker** (`nb3_worker`,
`R5_SLICES`, `LEVEL_CHANNELS`, etc.). Ela re-deriva a composição de canais de
cada nível a partir de listas brutas declaradas dentro da própria célula, e
recalcula os offsets de coluna do zero com a função `_offsets(ds, level)`.

Isso garante que a verificação é uma **fonte de verdade independente** — se o
worker tivesse um bug sistemático no fatiamento, esta célula o detectaria.

### SeizeIT2 não é verificado aqui

SeizeIT2 não participa da hierarquia R5→R0 (seus canais são completely diferentes).
Ele só tem R0, processado diretamente — não há R5 de referência para comparar.
A integridade do SeizeIT2 é verificada pela **Seção 8** (shape e NaN).

### Configuração

- `N_SAMPLE = 5` — verifica 5 gravações por dataset (rápido, ~segundos)
- `N_SAMPLE = None` — verifica todas (pode demorar vários minutos)

A comparação usa `np.allclose(atol=1e-6)` para tolerar diferenças mínimas de
arredondamento float32, mas na prática os arrays devem ser byte-a-byte idênticos
(o fatiamento não faz nenhuma operação aritmética).


In [ ]:
# ── Verificação cruzada R5 → níveis derivados ────────────────────────────────
# Completamente independente do código de extração.
# Recalcula offsets do zero e compara os arrays byte a byte.
# ─────────────────────────────────────────────────────────────────────────────

import os, glob, random
import numpy as np

FEAT_DIR_CC = os.path.join("data", "features")   # pasta a verificar
N_SAMPLE    = 5    # gravações por dataset a amostrar; None = todas

# ── Composição de canais (copiada aqui para ser independente do worker) ──────
_FRONTAL   = {"CHBMIT": ["FP1-F7","F7-T7","FP1-F3","FP2-F4","FP2-F8","F8-T8","FZ-CZ"],
               "Siena":  ["Fp1","F3","F7","Fz","Fp2","F4","F8"],
               "Mendeley":["Fp1","Fp2","F3","F4","F7","F8","Fz"]}
_TEMPORAL  = {"CHBMIT": ["T7-P7","T7-FT9","FT10-T8","T8-P8-0"],
               "Siena":  ["T3","T4","T5","T6"],
               "Mendeley":["T3","T4","T5","T6"]}
_CENTRAL   = {"CHBMIT": ["F3-C3","F4-C4"], "Siena": ["C3","C4"], "Mendeley": ["C3","C4"]}
_PARIETAL  = {"CHBMIT": ["P3-O1","P4-O2"], "Siena": ["P3","P4"], "Mendeley": ["P3","P4"]}
_OCCIPITAL = {"CHBMIT": ["P7-O1","P8-O2"], "Siena": ["O1","O2"], "Mendeley": ["O1","O2"]}
_R0_CORE   = {"CHBMIT": ["T7-FT9","FT10-T8"], "Siena": ["T3","T4"], "Mendeley": ["T3","T4"]}

_LEVEL_CH  = {}
for _ds in ["CHBMIT","Siena","Mendeley"]:
    _R5 = _FRONTAL[_ds]+_TEMPORAL[_ds]+_CENTRAL[_ds]+_PARIETAL[_ds]+_OCCIPITAL[_ds]
    _LEVEL_CH[(_ds,"R5")] = _R5
    _LEVEL_CH[(_ds,"R3")] = _FRONTAL[_ds]+_TEMPORAL[_ds]+_CENTRAL[_ds]
    _LEVEL_CH[(_ds,"R2")] = _FRONTAL[_ds]+_TEMPORAL[_ds]
    _LEVEL_CH[(_ds,"R1")] = _TEMPORAL[_ds]
    _LEVEL_CH[(_ds,"R0")] = _R0_CORE[_ds]

_NF = 19  # features por canal

def _offsets(ds, level):
    """Calcula (col_start, col_end) do nível dentro do vetor R5, do zero."""
    R5 = _LEVEL_CH[(ds, "R5")]
    ch = _LEVEL_CH[(ds, level)]
    idxs = [R5.index(c) for c in ch]
    return idxs[0] * _NF, (idxs[-1] + 1) * _NF

# ── Coleta pares (R5, nível_derivado) por dataset ────────────────────────────
print("=" * 70)
print(f"VERIFICAÇÃO CRUZADA R5 → DERIVADOS — {FEAT_DIR_CC}")
print("=" * 70)

erros   = []
ok_tot  = 0
chk_tot = 0

for ds in ["CHBMIT", "Siena", "Mendeley"]:
    # Lista todas as gravações R5 disponíveis para este dataset
    r5_files = sorted(glob.glob(
        os.path.join(FEAT_DIR_CC, "R5", f"{ds}__*__w*.npz")
    ))
    if not r5_files:
        print(f"\n{ds}: nenhum arquivo R5 encontrado — pulando.")
        continue

    # Amostra aleatória
    sample = random.sample(r5_files, min(N_SAMPLE or len(r5_files), len(r5_files)))
    print(f"\n{ds}: verificando {len(sample)} gravações de {len(r5_files)} disponíveis")

    for r5_path in sample:
        bn   = os.path.basename(r5_path)   # ex: CHBMIT__chb01__chb01_03__w30.npz
        stem = bn[len(ds)+2:]              # chb01__chb01_03__w30.npz
        # separa fkey e janela do nome do arquivo
        parts   = stem.rsplit("__w", 1)
        if len(parts) != 2:
            erros.append((bn, "nome de arquivo inesperado")); continue
        pat_fkey, w_str = parts          # "chb01__chb01_03", "30.npz"
        w_str = w_str.replace(".npz","")

        try:
            z5   = np.load(r5_path, allow_pickle=True)
            Xp5  = z5["X_pre"];  Xi5 = z5["X_inter"]
            z5.close()
        except Exception as e:
            erros.append((bn, f"erro ao ler R5: {e}")); continue

        # Verifica cada nível derivado
        for lv in ["R3", "R2", "R1", "R0"]:
            lv_path = os.path.join(
                FEAT_DIR_CC, lv, f"{ds}__{pat_fkey}__w{w_str}.npz"
            )
            chk_tot += 1

            if not os.path.exists(lv_path):
                erros.append((bn, f"{lv} não encontrado em disco")); continue

            try:
                zl  = np.load(lv_path, allow_pickle=True)
                Xpl = zl["X_pre"];  Xil = zl["X_inter"]
                zl.close()
            except Exception as e:
                erros.append((bn, f"erro ao ler {lv}: {e}")); continue

            # Calcula offsets esperados independentemente
            cs, ce = _offsets(ds, lv)

            # Compara X_pre
            if len(Xp5) > 0 and len(Xpl) > 0:
                expected_Xp = Xp5[:, cs:ce]
                if not np.array_equal(expected_Xp, Xpl):
                    # Diferença pode ser float32 vs float32 — verifica allclose
                    if not np.allclose(expected_Xp, Xpl, atol=1e-6):
                        diff = np.abs(expected_Xp - Xpl).max()
                        erros.append((bn, f"{lv} X_pre difere de R5[:,{cs}:{ce}] — max_diff={diff:.2e}"))
                        continue
            elif len(Xp5) != len(Xpl):
                erros.append((bn, f"{lv} X_pre: R5 tem {len(Xp5)} janelas, {lv} tem {len(Xpl)}"))
                continue

            # Compara X_inter
            if len(Xi5) > 0 and len(Xil) > 0:
                expected_Xi = Xi5[:, cs:ce]
                if not np.allclose(expected_Xi, Xil, atol=1e-6):
                    diff = np.abs(expected_Xi - Xil).max()
                    erros.append((bn, f"{lv} X_inter difere de R5[:,{cs}:{ce}] — max_diff={diff:.2e}"))
                    continue
            elif len(Xi5) != len(Xil):
                erros.append((bn, f"{lv} X_inter: R5 tem {len(Xi5)} janelas, {lv} tem {len(Xil)}"))
                continue

            ok_tot += 1

print()
print("=" * 70)
print(f"Pares verificados: {chk_tot}  |  OK: {ok_tot}  |  Erros: {len(erros)}")
if erros:
    print(f"\n⚠ ERROS ENCONTRADOS ({len(erros)}):")
    for fn, msg in erros:
        print(f"  {fn}: {msg}")
    print("\n→ Os arquivos com erro precisam ser removidos e reprocessados.")
    print("  Delete os .npz problemáticos e rode run_extraction() novamente.")
else:
    print("✓ Todos os níveis derivados são consistentes com R5.")
print("=" * 70)
